# 第11章 指示チューニング

## 11.2 指示チューニングの実装

### 11.2.1 環境の準備

In [1]:
!pip install datasets transformers[torch,sentencepiece] trl peft bitsandbytes

In [2]:
from transformers.trainer_utils import set_seed
set_seed(42)

### 11.2.2 データセットの準備

In [3]:
from pprint import pprint
from datasets import load_dataset

dataset = load_dataset("llm-book/oasst1-21k-ja", split="train")
pprint(dataset)

README.md:   0%|          | 0.00/355 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21164 [00:00<?, ? examples/s]

Dataset({
    features: ['conversation'],
    num_rows: 21164
})


In [4]:
from pprint import pprint

# pprintで見やすく表示する
pprint(dataset[0])

{'conversation': [{'content': 'こんにちは！', 'role': 'user'},
                  {'content': 'こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。',
                   'role': 'assistant'},
                  {'content': '世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。',
                   'role': 'user'},
                  {'content': '世界中の国をアルファベット順に並べたリストと、その国の推定人口です：\n'
                              '\n'
                              'アフガニスタン: 38,928,346 アルバニア: 2,877,797 '
                              'アルジェリア：44,344,744 アンドラ: 77,265 アンゴラ: 32,878,272 '
                              'アンティグア・バーブーダ: 97,929 アルゼンチン: 45,195,774 アルメニア: '
                              '2,977,600 オーストラリア: 25,499,884 オーストリア: 9,006,398 '
                              'アゼルバイジャン: 10,134,604 バハマ：393,248 バーレーン: '
                              '1,714,571 バングラデシュ: 164,689,383164,689,383 '
                              'バルバドス: 287,375 ベラルーシ: 9,449,323 ベルギー: '
                              '11,589,623 ベリーズ: 397,628 ベナン: 12,123,200 ブータン: 

### 11.2.3 チャットテンプレートの作成

In [5]:
from transformers import AutoTokenizer

base_model_name = "tokyotech-llm/Swallow-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/773 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/914k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

In [6]:
tokenizer.chat_template = """\
{%- for message in messages %}
{%- if message['role'] == 'user' %}
{{ bos_token + 'ユーザ：' + message['content'] + eos_token }}
{%- elif message['role'] == 'assistant' %}
{{ bos_token + 'アシスタント：'  + message['content'] + eos_token }}
{%- endif %}
{% if loop.last and add_generation_prompt %}
{{ bos_token + 'アシスタント：' }}
{%- endif %}
{% endfor %}\
"""

In [7]:
# デフォルトではトークナイズかつID化されたトークンのリストが返されるが、
# ここではtokenize=Falseとしてトークナイズ前の文字列を返すように設定
chat_text = tokenizer.apply_chat_template(
    dataset[0]["conversation"], tokenize=False
)
# 発話間に改行が含まれないため、見やすくするためにeos_tokenを改行に置換
print(chat_text.replace(tokenizer.eos_token, "\n"))

<s>ユーザ：こんにちは！
<s>アシスタント：こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。
<s>ユーザ：世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。
<s>アシスタント：世界中の国をアルファベット順に並べたリストと、その国の推定人口です：

アフガニスタン: 38,928,346 アルバニア: 2,877,797 アルジェリア：44,344,744 アンドラ: 77,265 アンゴラ: 32,878,272 アンティグア・バーブーダ: 97,929 アルゼンチン: 45,195,774 アルメニア: 2,977,600 オーストラリア: 25,499,884 オーストリア: 9,006,398 アゼルバイジャン: 10,134,604 バハマ：393,248 バーレーン: 1,714,571 バングラデシュ: 164,689,383164,689,383 バルバドス: 287,375 ベラルーシ: 9,449,323 ベルギー: 11,589,623 ベリーズ: 397,628 ベナン: 12,123,200 ブータン: 754,288 ボリビア: 11,673,095 ボスニア・ヘルツェゴビナ: 3,279,000 ボツワナ：2,335,814 ブラジル: 212,559,417 ブルネイ: 441,484 ブルガリア: 7,034,879 ブルキナファソ: 20,903,273 ブルンジ: 11,890,781 カボベルデ: 555,987 カンボジア: 16,205,218 カメルーン: 26,545,863 カナダ：37,742,154 中央アフリカ共和国4,829,767 チャド: 16,425,864 チリ: 19,116,201 中国: 1,439,323,776 コロンビア: 50,882,891 コモロ: 869,601 コンゴ民主共和国：87,534,403 コンゴ共和国5,457,821 コスタリカ5,094,118 コートジボワール: 26,378,274 クロアチア: 4,105,267 キューバ: 11,239,224 キプロス：1,207,359 チェコ：10,708,919 デンマーク：5,792,2025,792,202



In [8]:
# 会話データの末尾のアシスタントの発話を除き、生成を促すための文字列を追加
chat_text = tokenizer.apply_chat_template(
    dataset[0]["conversation"][:-1],
    tokenize=False,
    add_generation_prompt=True,
)
print(chat_text.replace(tokenizer.eos_token, "\n"))

<s>ユーザ：こんにちは！
<s>アシスタント：こんにちは！ご質問やお困りのことがありましたら、何でもご相談ください。何が必要か教えてください。
<s>ユーザ：世界のすべての国をアルファベット順に、それぞれの国の人口を教えてください。
<s>アシスタント：


### 11.2.4 トークンIDへの変換

In [10]:
# チャットテンプレートを適用してトークンIDに変換
tokenized_dataset = [
    tokenizer.apply_chat_template(item["conversation"])
    for item in dataset
]

トークンID: {'input_ids': [1, 39944, 30383, 33328, 30584, 2, 1, 36166, 32113, 32040, 30383, 33328, 30584, 31622, 32425, 32099, 31111, 30697, 36451, 30199, 32002, 30458, 32009, 32006, 32075, 30330, 31502, 39682, 31622, 32277, 32059, 30267, 31502, 30458, 32090, 30412, 32903, 30466, 32059, 30267, 2, 1, 39944, 30383, 32198, 30199, 32668, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 30330, 32728, 30199, 30356, 30199, 35620, 30396, 32903, 30466, 32059, 30267, 2, 1, 36166, 32113, 32040, 30383, 32198, 30275, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 37231, 30366, 33634, 30364, 30330, 32016, 30356, 30199, 40875, 35620, 32001, 30383, 0, 34874, 41714, 39676, 29901, 0, 29941, 29947, 29892, 29929, 29906, 29947, 29892, 29941, 29946, 29953, 0, 33997, 32873, 29901, 0, 29906, 29892, 29947, 29955, 29955, 29892, 29955, 29929, 29955, 0, 32513, 32882, 32166, 30383, 29946, 29946, 29892, 29941, 29946, 29946, 29892, 29955, 29946, 29946, 0, 32709, 32498, 29901, 0, 29955, 29955, 29892, 2990

In [11]:
# トークンIDに変換されたデータセットの先頭を表示
token_ids = tokenized_dataset[0]
print("トークンID:", token_ids)
print("トークン:", tokenizer.convert_ids_to_tokens(token_ids["input_ids"]))

トークンID: {'input_ids': [1, 39944, 30383, 33328, 30584, 2, 1, 36166, 32113, 32040, 30383, 33328, 30584, 31622, 32425, 32099, 31111, 30697, 36451, 30199, 32002, 30458, 32009, 32006, 32075, 30330, 31502, 39682, 31622, 32277, 32059, 30267, 31502, 30458, 32090, 30412, 32903, 30466, 32059, 30267, 2, 1, 39944, 30383, 32198, 30199, 32668, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 30330, 32728, 30199, 30356, 30199, 35620, 30396, 32903, 30466, 32059, 30267, 2, 1, 36166, 32113, 32040, 30383, 32198, 30275, 30199, 30356, 30396, 32513, 32148, 39383, 33045, 30353, 37231, 30366, 33634, 30364, 30330, 32016, 30356, 30199, 40875, 35620, 32001, 30383, 0, 34874, 41714, 39676, 29901, 0, 29941, 29947, 29892, 29929, 29906, 29947, 29892, 29941, 29946, 29953, 0, 33997, 32873, 29901, 0, 29906, 29892, 29947, 29955, 29955, 29892, 29955, 29929, 29955, 0, 32513, 32882, 32166, 30383, 29946, 29946, 29892, 29941, 29946, 29946, 29892, 29955, 29946, 29946, 0, 32709, 32498, 29901, 0, 29955, 29955, 29892, 2990

In [12]:
tokenizer.pad_token = tokenizer.unk_token

In [17]:
from trl import DataCollatorForCompletionOnlyLM

bos = tokenizer.bos_token
collator = DataCollatorForCompletionOnlyLM(
    instruction_template=bos + "ユーザ:",      # ユーザの発話開始を示す文字列
    response_template=bos + "アシスタント：", # アシスタントの返答開始を示す文字列
    tokenizer=tokenizer,  # トークナイザ
)
# トークナイズされたデータセットの先頭をミニバッチ構築処理
batch = collator([tokenized_dataset[1]])
input_ids = batch["input_ids"][0]
labels = batch["labels"][0]

ImportError: cannot import name 'DataCollatorForCompletionOnlyLM' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [18]:
!pip show trl

Name: trl
Version: 1.5.1
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, jinja2, packaging, transformers
Required-by: 


In [19]:
# 互換性のあるバージョン（例: 0.8.6）を指定して再インストールします。
# インストール完了後は、必ず上部メニューの「ランタイム」>「セッションを再起動」を実行してください。
!pip install --upgrade trl==0.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 1.5.1
    Uninstalling trl-1.5.1:
      Successfully uninstalled trl-1.5.1


In [ ]:
print("入力トークンID:", input_ids)
print("正解ラベル:", labels)